## Setup

In [1]:
import re

print("ready")

ready


## The library — provided

The responses and the text-cleaning function. Nothing here needs changing.

In [2]:
RESPONSES = {
    'hours'     : 'We are open 08:00 to 20:00 Monday to Friday, and 09:00 to 13:00 on Saturday. Closed on Sunday.',
    'borrow'    : 'You may borrow up to 4 books at a time, for 14 days.',
    'renew'     : 'A loan can be renewed twice online, unless another reader has reserved the book.',
    'fines'     : 'Overdue books are 200 RWF per book per day, capped at 5,000 RWF.',
    'printing'  : 'Printing is 50 RWF per page in black and white, 200 RWF in colour.',
    "rooms":      "(handled by the booking flow in Part B)",
}

FALLBACK = (
    "Sorry, I cannot help with that. I can answer questions about opening hours, "
    "borrowing, renewals, fines, study rooms and printing. "
    "Please ask at the desk for anything else."
)


def normalise(text):
    """Lower case, trimmed, punctuation removed - colons kept, for times."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


print(len(RESPONSES), "responses loaded")

6 responses loaded


## The test set — provided

The eighteen sample phrasings from the brief, plus capitals, a misspelling, a time, a number, three out-of-scope requests, and three messages that are expected to fail. Do not edit it.

In [3]:
TESTS = [
    ('what time do you open', 'hours'),
    ('are you open on saturday', 'hours'),
    ('till what time on friday', 'hours'),
    ('how many books can i take', 'borrow'),
    ('borrowing limit?', 'borrow'),
    ('how long can i keep a book', 'borrow'),
    ('can i renew', 'renew'),
    ('renewing my loan', 'renew'),
    ('i want to extend my book', 'renew'),
    ('how much is the fine', 'fines'),
    ('what are the overdue charges', 'fines'),
    ('i returned it late what now', 'fines'),
    ('i want to book a study room', 'rooms'),
    ('can i reserve a room', 'rooms'),
    ('study room booking', 'rooms'),
    ('do you have printing', 'printing'),
    ('how much per page', 'printing'),
    ('can i photocopy', 'printing'),
    ('WHAT TIME DO YOU OPEN', 'hours'),
    ('opening hours?', 'hours'),
    ('are you open at 18:00', 'hours'),
    ('how mny books', 'borrow'),
    ('renewals', 'renew'),
    ('fine is how much', 'fines'),
    ('booking a room for 3 people', 'rooms'),
    ('printing costs', 'printing'),
    ('can you write my assignment', 'fallback'),
    ('do you sell coffee', 'fallback'),
    ('what is the wifi password', 'fallback'),
    ('asdfgh', 'fallback'),
    ('ni angahe for printing', 'printing'),
    ('do you have a bookshop', 'fallback'),
    ('how many books can i take and what are the fines', 'borrow+fines'),
    ('rnw my bk', 'renew'),
    ('can i extend my deadline', 'fallback'),
]

print(len(TESTS), "test messages loaded")

35 test messages loaded


---

## Part A — The pattern table  [40 Points]

One regular expression per intent. Each must catch all three of its own sample phrasings, catch
nothing belonging to another intent, and be wrapped in `\b`.

**The ordering trap.** `"i want to book a study room"` contains the word *book*, which also belongs
to `borrow`. Since the first match wins, `rooms` must come first. Decide the whole order
deliberately — you explain it in the README.

In [4]:
RULES = [
    ("rooms",    r"TODO"),
    ("hours",    r"TODO"),
    ("borrow",   r"TODO"),
    ("renew",    r"TODO"),
    ("fines",    r"TODO"),
    ("printing", r"TODO"),
]


def match(message):
    """PROVIDED. Return the tag of the first pattern that matches, or "fallback"."""
    text = normalise(message)
    for tag, pattern in RULES:
        if re.search(pattern, text):
            return tag
    return "fallback"


for m in ["what time do you open", "can i renew", "i want to book a study room",
          "do you have a bookshop"]:
    print(f"{m!r:36} -> {match(m)}")

'what time do you open'              -> fallback
'can i renew'                        -> fallback
'i want to book a study room'        -> fallback
'do you have a bookshop'             -> fallback


---

## Part B — The study room booking flow  [40 Points]

Three values collected one at a time, then confirmed. The `day` step is written for you as a model;
copy its shape for the other two.

Each step needs a **re-ask path**: if the answer is not recognised, ask that question again rather
than storing the value or moving on.

The party size is a *number*, so a keyword list cannot validate it. `SIZE` uses `\b([1-8])\b`,
which both matches and rejects — 9 and 0 will not match.

In [5]:
DAYS  = r"\b(mon|tue|wed|thu|fri|sat)\w*\b"
SLOTS = r"\b(morning|afternoon|evening)\b"
SIZE  = r"\b([1-8])\b"

flow = {"step": None, "collected": {}}


def reset_flow():
    """PROVIDED. Abandon the booking and clear what was collected."""
    flow["step"] = None
    flow["collected"] = {}


def start_booking():
    """PROVIDED. Begin the flow and ask the first question."""
    flow["step"] = "day"
    flow["collected"] = {}
    return "Which day would you like the room? Monday to Saturday."


def continue_booking(message):
    """Handle one answer inside the booking flow."""
    text = normalise(message)

    if flow["step"] == "day":
        found = re.search(DAYS, text)
        if found:
            flow["collected"]["day"] = found.group(1)
            flow["step"] = "time"
            return "Which time? Morning, afternoon or evening."
        return "Which day? Monday to Saturday."

    if flow["step"] == "time":
        # TODO: if SLOTS matches, store group(1), set step to "size", and ask
        #       "How many people? Between 1 and 8."   Otherwise re-ask this step.
        pass

    if flow["step"] == "size":
        # TODO: if SIZE matches, store group(1); then build the confirmation from
        #       flow["collected"], call reset_flow(), and return the confirmation.
        #       Otherwise re-ask this step.
        pass

    return "Something went wrong. Starting again."


# Non-interactive walk-through
print("Bot:", start_booking())
for answer in ["thursday", "afternoon", "4"]:
    print("You:", answer)
    print("Bot:", continue_booking(answer))

Bot: Which day would you like the room? Monday to Saturday.
You: thursday
Bot: Which time? Morning, afternoon or evening.
You: afternoon
Bot: Something went wrong. Starting again.
You: 4
Bot: Something went wrong. Starting again.


In [6]:
EXIT_WORDS = ("bye", "goodbye", "exit", "quit", "thanks", "murakoze")


def reply_to(message):
    """One turn of the library chatbot."""
    if normalise(message) in EXIT_WORDS:
        reset_flow()
        return "Thank you. Murakoze."

    if flow["step"] is not None:
        return continue_booking(message)

    tag = match(message)
    if tag == "rooms":
        return start_booking()
    if tag == "fallback":
        return FALLBACK
    return RESPONSES[tag]


for m in ["how much is the fine", "i want to book a study room", "friday", "bye"]:
    print(f"You: {m}")
    print(f"Bot: {reply_to(m)}")
    print()

You: how much is the fine
Bot: Something went wrong. Starting again.

You: i want to book a study room
Bot: Something went wrong. Starting again.

You: friday
Bot: Something went wrong. Starting again.

You: bye
Bot: Thank you. Murakoze.



### Try it

Run the cell below **on its own**. Book a room, then try abandoning one halfway through with `bye`.

In [ ]:
def chat():
    """The full chatbot. Run this cell on its own - it waits for input."""
    print("Campus Library assistant - an automated service, not a librarian.")
    print("I can help with hours, borrowing, renewals, fines, study rooms and printing.")
    print("Type 'bye' to finish.")
    print()
    while True:
        message = input("You: ")
        print("Bot:", reply_to(message))
        print()
        if normalise(message) in EXIT_WORDS:
            break


chat()

Campus Library assistant - an automated service, not a librarian.
I can help with hours, borrowing, renewals, fines, study rooms and printing.
Type 'bye' to finish.



---

## Part C — Test and document  [20 Points]

Run the test set and report what it gives. **Three of the messages are expected to fail** — they are
there so that the diagnosis has something to work on. Do not edit the tests to improve the figure.

In [ ]:
correct, wrong = 0, []
for message, expected in TESTS:
    got = match(message)
    if got == expected:
        correct += 1
    else:
        wrong.append((message, expected, got))

print(f"{correct} of {len(TESTS)} messages returned the expected tag")
print()
print("Failures to account for:")
for m, e, g in wrong:
    print(f"  {m!r:52} expected {e:14} got {g}")

### Written answers

Three short answers. These carry marks — they are where you show the failures are understood rather than merely listed.

In [ ]:
answers = """
1. Rule ordering. Which two intents collided, and why must rooms be checked
   before borrow?


2. For each failure printed above, give the cause: keyword too short, missing
   word boundary, or a word genuinely shared between two intents.


3. Which failure could not be fixed by any pattern at all, and which later week
   of this module deals with it?

"""

print(answers)